# 11 - Orchestrate Rerunnable Deployment

Runs the Fabric notebooks in dependency order through the supported on-demand item job API. It supports dry-run, checkpoint/resume, force, optional serving/BI deployment, and a required second deterministic pass. A failed or unsupported required job stops the orchestration; no success is inferred from submission alone.

In [ ]:
# PARAMETERS
workspace_id = ''
capacity_reference = ''
orchestration_run_id = ''
environment_name = 'dev'
resource_prefix = 'fao-demo'
simulation_profile = 'smoke'
deployment_mode = 'dry-run'  # dry-run | plan | apply
resume_from_checkpoint = True
force = False
run_second_pass = True
include_lakehouse_maintenance = False
include_platform_deployment = False
deploy_conditional_artifacts = False
artifact_root = '/lakehouse/default/Files/airport-ops-mvp'
warehouse_sql_endpoint = ''
kql_query_uri = ''
job_timeout_seconds = 1800

import re
import time
import uuid
from datetime import datetime, timezone

import requests

API_BASE = 'https://api.fabric.microsoft.com/v1'


In [ ]:
dry_run = deployment_mode in {'dry-run', 'plan'}
RUN_ID = orchestration_run_id or ('ORCH-' + uuid.uuid4().hex[:16].upper())
CHECKPOINTS = []

assert deployment_mode in {'dry-run', 'plan', 'apply'}
assert environment_name in {'dev', 'test'}
assert resource_prefix == 'fao-demo' and re.fullmatch(r'[a-z0-9-]{3,24}', resource_prefix)
assert simulation_profile in {'unit', 'smoke', 'demo', 'enterprise'}
if deployment_mode == 'apply':
    assert re.fullmatch(r'[0-9a-fA-F-]{36}', workspace_id), 'A runtime Fabric workspace GUID is required'
    assert capacity_reference, 'A runtime capacity reference is required'

DATA_NOTEBOOKS = [
    '01_Generate_Sample_Data','02_Bronze_to_Silver','03_Silver_to_Gold',
    '04_Generate_Physical_Spatial_Context','05_Build_Agent_Ontology_Context',
    '07_Generate_Enterprise_Bronze','08_Enterprise_Bronze_to_Silver','09_Enterprise_Silver_to_Gold']


def checkpoint(phase, notebook_name, status, detail='', job_instance_id=''):
    row = {
        'orchestration_run_id':RUN_ID,'environment_name':environment_name,
        'resource_prefix':resource_prefix,'deployment_mode':deployment_mode,
        'simulation_profile':simulation_profile,'phase':phase,'notebook_name':notebook_name,
        'status':status,'status_detail':detail[:4000],'job_instance_id':job_instance_id or '',
        'observed_at':datetime.now(timezone.utc),'is_synthetic':True}
    CHECKPOINTS.append(row)
    print(status,phase,notebook_name,detail)
    try:
        spark.createDataFrame([row]).write.mode('append').format('delta').saveAsTable('orchestration_checkpoint')
    except Exception as exc:
        print('Checkpoint Delta write unavailable:',str(exc))
    return row


In [ ]:
def auth_headers():
    return {'Authorization': 'Bearer ' + notebookutils.credentials.getToken('pbi'), 'Content-Type': 'application/json'}


def notebook_items():
    response = requests.get(API_BASE + '/workspaces/' + workspace_id + '/items?type=Notebook', headers=auth_headers(), timeout=90)
    if response.status_code != 200:
        raise RuntimeError('Notebook discovery failed: ' + response.text[:4000])
    return {item['displayName']: item for item in response.json().get('value', [])}


def parameter_payload(parameters):
    if not parameters:
        return None
    values = {}
    for name, value in parameters.items():
        parameter_type = 'bool' if isinstance(value, bool) else 'string'
        parameter_value = str(value).lower() if isinstance(value, bool) else str(value)
        values[name] = {'value': parameter_value, 'type': parameter_type}
    return {'executionData': {'parameters': values}}


def already_succeeded(phase, notebook_name):
    if force or not resume_from_checkpoint:
        return False
    try:
        if not spark.catalog.tableExists('orchestration_checkpoint'):
            return False
        return spark.table('orchestration_checkpoint').filter(
            (spark.table('orchestration_checkpoint').orchestration_run_id == RUN_ID) &
            (spark.table('orchestration_checkpoint').phase == phase) &
            (spark.table('orchestration_checkpoint').notebook_name == notebook_name) &
            (spark.table('orchestration_checkpoint').status == 'SUCCEEDED')
        ).limit(1).count() == 1
    except Exception:
        return False


def run_notebook(phase, notebook_name, item_map, parameters=None):
    if already_succeeded(phase, notebook_name):
        checkpoint(phase, notebook_name, 'SKIPPED_CHECKPOINT', 'Previously completed for this orchestration run')
        return
    if dry_run:
        checkpoint(phase, notebook_name, 'DRY_RUN', 'Would submit and poll notebook job')
        return
    if notebook_name not in item_map:
        checkpoint(phase, notebook_name, 'FAILED', 'Notebook item was not found in the workspace')
        raise KeyError(notebook_name)
    item_id = item_map[notebook_name]['id']
    response = requests.post(
        API_BASE + '/workspaces/' + workspace_id + '/items/' + item_id + '/jobs/instances?jobType=RunNotebook',
        headers=auth_headers(), json=parameter_payload(parameters), timeout=90,
    )
    if response.status_code != 202 or not response.headers.get('Location'):
        checkpoint(phase, notebook_name, 'FAILED', 'Job submission failed: ' + response.text[:4000])
        raise RuntimeError('Job submission failed for ' + notebook_name)
    location = response.headers['Location']
    job_instance_id = location.rstrip('/').split('/')[-1]
    checkpoint(phase, notebook_name, 'SUBMITTED', 'Notebook job accepted', job_instance_id)
    started = time.time()
    while time.time() - started < job_timeout_seconds:
        status_response = requests.get(location, headers=auth_headers(), timeout=90)
        if status_response.status_code != 200:
            checkpoint(phase, notebook_name, 'FAILED', status_response.text[:4000], job_instance_id)
            raise RuntimeError('Job polling failed for ' + notebook_name)
        body = status_response.json()
        status = str(body.get('status', '')).lower()
        if status in {'completed', 'succeeded'}:
            checkpoint(phase, notebook_name, 'SUCCEEDED', 'Notebook job completed', job_instance_id)
            return
        if status in {'failed', 'cancelled', 'deduped'}:
            checkpoint(phase, notebook_name, 'FAILED', str(body.get('failureReason', body))[:4000], job_instance_id)
            raise RuntimeError('Notebook job failed for ' + notebook_name)
        time.sleep(int(status_response.headers.get('Retry-After', '5')))
    checkpoint(phase, notebook_name, 'FAILED', 'Notebook job timed out', job_instance_id)
    raise TimeoutError(notebook_name + ' exceeded ' + str(job_timeout_seconds) + ' seconds')

In [ ]:
items = {} if dry_run else notebook_items()

run_notebook(
    'PREFLIGHT','00_Validate_Prerequisites',items,
    {'workspace_id':workspace_id,'capacity_reference':capacity_reference,'artifact_root':artifact_root,
     'environment_name':environment_name,'dry_run':dry_run,'strict_mode':True})

for notebook_name in DATA_NOTEBOOKS:
    parameters={'artifact_root':artifact_root,'simulation_profile_override':simulation_profile} if notebook_name=='01_Generate_Sample_Data' else None
    run_notebook('FIRST_PASS_DATA',notebook_name,items,parameters)
run_notebook('FIRST_PASS_VALIDATION','06_Validate_Extended_MVP',items)
run_notebook('FIRST_PASS_VALIDATION','12_Validate_Production_Demo',items,
             {'require_second_run':False,'validation_phase':'BASELINE'})

if run_second_pass:
    for notebook_name in DATA_NOTEBOOKS:
        parameters={'artifact_root':artifact_root,'simulation_profile_override':simulation_profile} if notebook_name=='01_Generate_Sample_Data' else None
        run_notebook('SECOND_PASS_DATA',notebook_name,items,parameters)
    run_notebook('SECOND_PASS_VALIDATION','06_Validate_Extended_MVP',items)
    run_notebook('SECOND_PASS_VALIDATION','12_Validate_Production_Demo',items,
                 {'require_second_run':True,'validation_phase':'SECOND_RUN'})

if include_lakehouse_maintenance:
    run_notebook('LAKEHOUSE_MAINTENANCE','16_Lakehouse_Maintenance',items,
                 {'dry_run':dry_run,'apply_table_properties':True,'apply_optimize':False,
                  'apply_vacuum':False,'vacuum_retention_hours':'168'})

if include_platform_deployment:
    run_notebook(
        'PLATFORM_DEPLOYMENT','10_Deploy_Platform_Artifacts',items,
        {'workspace_id':workspace_id,'artifact_root':artifact_root,
         'warehouse_sql_endpoint':warehouse_sql_endpoint,'kql_query_uri':kql_query_uri,
         'environment_name':environment_name,'dry_run':dry_run,
         'deploy_conditional_artifacts':deploy_conditional_artifacts})

run_notebook('STATUS','15_Deployment_Status',items)
failures=[row for row in CHECKPOINTS if row['status']=='FAILED']
assert not failures,'Orchestration failed in '+str(len(failures))+' steps'
print('Orchestration',RUN_ID,'completed with',len(CHECKPOINTS),'checkpoint events; dry_run=',dry_run)